# 实验5.4 香橙派开发板OM模型转换与手写数字识别

> **实验类型**：ONNX → ATC转换 → OM推理 · 昇腾香橙派端侧部署实验  
> **目标硬件**：香橙派开发板（昇腾310B NPU，SOC 为 Ascend310B4）  
> **前置实验**：实验5.3（产生 ONNX 模型文件）  
> **运行环境说明**：本 Notebook 描述在香橙派开发板上的部署与运行流程，**不在 PC 端仿真运行**

---

## 实验概述

本实验以 MNIST 手写数字识别任务为载体，在**香橙派开发板**（昇腾310B NPU）上完成模型部署的最后两个环节：**ATC 模型转换**和**ACL 端侧推理**。实验5.3已在云沙箱（昇腾910B3）上完成模型训练并导出 ONNX 文件，本实验将这些 ONNX 文件部署到香橙派上，通过 ATC 编译为 OM 离线模型，再利用 AscendCL 接口在 NPU 上执行推理，识别 `output/test_images/` 目录下的手写数字测试图片。

```text
实验5.3 (云沙箱 910B3)                    本实验 (香橙派 310B4)
┌──────────────────────┐               ┌──────────────────────────────┐
│ PyTorch训练          │               │  ATC模型转换 (ONNX → OM)     │
│ ONNX导出             │ ──ONNX文件──> │  ACL端侧推理 (OM → NPU推理)  │
│ 测试图片生成         │ ──测试图片──> │  手写数字识别与成功率统计     │
└──────────────────────┘               └──────────────────────────────┘
```

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;"><strong>实验名称</strong></td>
<td style="text-align: left;">香橙派OM模型转换与手写数字识别</td>
</tr>
<tr>
<td style="text-align: left;"><strong>目标硬件</strong></td>
<td style="text-align: left;">香橙派开发板（昇腾310B NPU，Ascend310B4）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>软件环境</strong></td>
<td style="text-align: left;">CANN Toolkit · AscendCL (ACL) · Python 3</td>
</tr>
<tr>
<td style="text-align: left;"><strong>输入文件</strong></td>
<td style="text-align: left;"><code>models/*.onnx</code>（实验5.3产出）、<code>output/test_images/*.png</code>（实验5.3产出）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>输出文件</strong></td>
<td style="text-align: left;"><code>output/*.om</code>（OM离线模型）、<code>output/test_images/batch_result.json</code>（识别结果）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>源代码目录</strong></td>
<td style="text-align: left;"><code>code/</code>（所有 Python 程序和脚本）</td>
</tr>
</table>

> **重要说明**：本 Notebook **不在 PC 端仿真运行**，而是详细描述在香橙派开发板上的运行程序和操作步骤。所有代码已放在 `code/` 目录下，可直接拷贝到香橙派上运行。

---

## 实验目标

- **知识目标**：理解 ATC 模型编译优化原理（算子映射、图融合、内存规划）；理解 AscendCL 推理编程模型（初始化→加载→推理→释放七阶段）；理解端侧部署的完整流程。
- **能力目标**：能够在香橙派开发板上正确使用 ATC 工具完成 ONNX→OM 转换；能够运行基于 ACL 的 OM 推理程序识别手写数字；能够统计识别成功率并分析结果。
- **素养目标**：形成"云训练→端部署"的完整工程思维，理解云沙箱与边缘设备的协作关系。

---

## 实验原理

### 1. 端侧部署链路

```text
ONNX模型 (.onnx)  →  ATC编译  →  昇腾OM模型 (.om)  →  ACL推理  →  识别结果
  跨框架中间格式      模型编译优化     NPU离线推理模型     NPU执行     数字0~9
```

**链路各环节作用**：
- **ONNX**：实验5.3在云沙箱上训练 SimpleCNN 并导出的跨框架中间格式模型，与训练框架解耦，可跨平台部署。
- **ATC**：昇腾模型编译工具，将 ONNX 编译为针对 Ascend310B4 硬件优化的 OM 离线模型。编译过程完成算子映射到 NPU AICore、图融合（如 Conv+BN+ReLU 融合）、内存复用规划等硬件优化。
- **OM**：昇腾离线推理模型格式，包含预编译的 NPU 指令流，推理时直接在 AICore 上执行，无需运行时图解释。
- **ACL**：AscendCL 推理编程接口，负责加载 OM 模型、管理 Host/Device 内存、执行推理并返回结果。

### 2. ATC 编译优化三大核心

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">优化类型</th>
<th style="text-align: left;">说明</th>
<th style="text-align: left;">效果</th>
</tr>
<tr>
<td style="text-align: left;">算子调度优化</td>
<td style="text-align: left;">将算子映射到 NPU 的 AICore，选择最优实现</td>
<td style="text-align: left;">矩阵乘加由专用电路完成</td>
</tr>
<tr>
<td style="text-align: left;">图融合</td>
<td style="text-align: left;">将相邻算子融合为单算子（如 Conv+BN+ReLU）</td>
<td style="text-align: left;">减少中间结果内存搬运</td>
</tr>
<tr>
<td style="text-align: left;">内存复用</td>
<td style="text-align: left;">分析张量生命周期，让不重叠的中间结果复用内存</td>
<td style="text-align: left;">减少总内存占用</td>
</tr>
</table>

### 3. ACL 推理七阶段

```text
acl.init()          阶段1: ACL初始化
acl.rt.set_device()  阶段2: 设备设置与上下文创建
acl.mdl.load_from_mem()  阶段3: 加载OM模型到Device内存
acl.mdl.get_desc()      阶段4: 查询输入/输出规格
acl.rt.malloc()         阶段5: 申请Device侧输入/输出内存
acl.rt.memcpy(H2D)      阶段6: Host数据拷贝到Device
acl.mdl.execute()       阶段7: 执行NPU推理
acl.rt.memcpy(D2H)      阶段8: 结果拷贝回Host
acl.rt.free() / acl.finalize()  阶段9: 释放所有资源
```

![ACL推理流程](../../images/acl_runtime.jpg)

**ACL 推理流程解读**：ACL 推理遵循严格的资源管理流程。初始化阶段设置设备和工作上下文；加载阶段将 OM 文件读入内存并注册为模型；查询阶段获取输入/输出大小以分配正确大小的内存；数据准备阶段在 Device 侧分配内存并将 Host 数据拷贝过去；执行阶段提交推理任务并等待完成；最后必须按逆序释放所有资源以避免内存泄漏。

---

## 步骤一：香橙派开发板环境准备

> **运行环境**：香橙派开发板（Ascend310B4），需已安装 CANN Toolkit。

### 1.1 硬件与软件要求

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">要求</th>
</tr>
<tr>
<td style="text-align: left;">硬件</td>
<td style="text-align: left;">香橙派 AIPro 开发板（昇腾310B NPU，SOC: Ascend310B4）</td>
</tr>
<tr>
<td style="text-align: left;">操作系统</td>
<td style="text-align: left;">Ubuntu 22.04 (ARM64)</td>
</tr>
<tr>
<td style="text-align: left;">CANN Toolkit</td>
<td style="text-align: left;">已安装并配置环境变量</td>
</tr>
<tr>
<td style="text-align: left;">Python</td>
<td style="text-align: left;">3.9 或 3.10</td>
</tr>
<tr>
<td style="text-align: left;">依赖库</td>
<td style="text-align: left;">numpy (<2), Pillow</td>
</tr>
</table>

### 1.2 环境初始化

在香橙派终端中执行以下命令，加载 CANN 环境变量：

```bash
# 方式一：使用 code 目录下的环境初始化脚本
cd code
source setup_env.sh

# 方式二：手动加载 CANN 环境
source /usr/local/Ascend/ascend-toolkit/set_env.sh

# 限制 ATC 并行编译进程数，防止开发板内存耗尽（香橙派内存通常 1-2GB）
export TE_PARALLEL_COMPILER=1
export MAX_COMPILE_CORE_NUMBER=1
```

### 1.3 验证环境

```bash
# 检查 NPU 设备信息（确认 SOC 版本为 Ascend310B4）
npu-smi info

# 检查 ATC 工具是否可用
atc --help | head -5

# 检查 Python 依赖
python3 -c "import numpy; print('numpy', numpy.__version__)"
python3 -c "import acl; print('acl loaded')"
```

**预期输出**：
- `npu-smi info` 显示芯片型号为 Ascend310B4
- `atc` 命令可执行并显示参数帮助
- `import acl` 成功，说明 CANN Python 接口已配置

### 1.4 安装 Python 依赖

```bash
cd code
pip3 install -r requirements.txt
```

---

## 步骤二：项目文件传输与目录结构

> 将实验5.3产出的 ONNX 模型和测试图片从 PC 传输到香橙派开发板。

### 2.1 需要传输的文件

实验5.3在云沙箱上产出以下文件，需传输到香橙派：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">说明</th>
<th style="text-align: left;">传输到</th>
</tr>
<tr>
<td style="text-align: left;"><code>models/simplecnn_mnist_fp32.onnx</code></td>
<td style="text-align: left;">FP32 全精度 ONNX 模型</td>
<td style="text-align: left;"><code>models/</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>models/simplecnn_mnist_pruned.onnx</code></td>
<td style="text-align: left;">剪枝优化 ONNX 模型</td>
<td style="text-align: left;"><code>models/</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>output/test_images/*.png</code></td>
<td style="text-align: left;">20张手写数字测试图片</td>
<td style="text-align: left;"><code>output/test_images/</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>output/test_images/test_meta.json</code></td>
<td style="text-align: left;">测试图片元数据</td>
<td style="text-align: left;"><code>output/test_images/</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>code/</code></td>
<td style="text-align: left;">推理程序代码目录</td>
<td style="text-align: left;"><code>code/</code></td>
</tr>
</table>

### 2.2 文件传输方法

```bash
# 方式一：使用 scp 从 PC 传输到香橙派
scp -r models/ user@orangepi:~/Lab5_4/
scp -r output/test_images/ user@orangepi:~/Lab5_4/output/
scp -r code/ user@orangepi:~/Lab5_4/

# 方式二：使用 U盘拷贝
# 方式三：使用共享文件夹（NFS/Samba）
```

### 2.3 香橙派上的目录结构

```text
Lab5_4/
├── models/                          # 实验5.3产出的 ONNX 模型
│   ├── simplecnn_mnist_fp32.onnx    # FP32 全精度模型 (~1.6MB)
│   └── simplecnn_mnist_pruned.onnx  # 剪枝优化模型 (~0.8MB)
├── output/                          # OM模型输出与测试图片
│   ├── simplecnn_mnist_fp32.om      # ATC转换产出的OM模型（本实验生成）
│   ├── simplecnn_mnist_pruned.om    # ATC转换产出的OM模型（本实验生成）
│   └── test_images/                 # 实验5.3产出的测试图片
│       ├── test_00_label0.png       # 20张手写数字图片
│       ├── test_01_label0.png
│       ├── ... 
│       ├── test_19_label9.png
│       ├── test_meta.json           # 测试图片元数据
│       └── batch_result.json        # 批量识别结果（本实验生成）
├── code/                          # 推理程序代码（本实验核心）
│   ├── setup_env.sh                 # 环境初始化脚本
│   ├── atc_convert.sh               # ATC模型转换脚本
│   ├── acl_classifier.py             # ACL推理分类器封装
│   ├── preprocess.py                # 图像预处理模块
│   ├── infer_single.py              # 单张图片识别脚本
│   ├── infer_batch.py               # 批量图片识别脚本
│   ├── requirements.txt             # Python依赖列表
│   └── README.md                    # code目录使用说明
└── lab5.4_orangepi_om_model_conversion.ipynb  # 本实验文档
```

---

## 步骤三：code 目录代码功能详解

> 所有在香橙派上运行的程序均基于 Python 实现，统一放在 `code/` 目录下。以下详细说明每个文件的功能。

### 3.1 `setup_env.sh` — 环境初始化脚本

**功能**：加载 CANN Toolkit 环境变量，设置 ATC 并行编译限制，配置 ACL Python 接口路径。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">配置项</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>source set_env.sh</code></td>
<td style="text-align: left;">加载 CANN 工具链环境变量（ATC、ACL 等依赖）</td>
</tr>
<tr>
<td style="text-align: left;"><code>TE_PARALLEL_COMPILER=1</code></td>
<td style="text-align: left;">限制 ATC 单进程编译，防止开发板内存耗尽</td>
</tr>
<tr>
<td style="text-align: left;"><code>MAX_COMPILE_CORE_NUMBER=1</code></td>
<td style="text-align: left;">限制编译核心数为1</td>
</tr>
<tr>
<td style="text-align: left;"><code>PYTHONPATH</code></td>
<td style="text-align: left;">配置 ACL Python 模块搜索路径</td>
</tr>
</table>

**使用方法**：
```bash
cd code
source setup_env.sh
```

### 3.2 `atc_convert.sh` — ATC 模型转换脚本

**功能**：将 `models/` 目录下的 ONNX 模型通过 ATC 命令转换为 OM 模型，输出到 `output/` 目录。自动转换 FP32 和剪枝两个模型。

**ATC 命令核心参数**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">值</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>--framework</code></td>
<td style="text-align: left;">5</td>
<td style="text-align: left;">输入格式为 ONNX</td>
</tr>
<tr>
<td style="text-align: left;"><code>--model</code></td>
<td style="text-align: left;"><code>models/*.onnx</code></td>
<td style="text-align: left;">输入 ONNX 模型路径</td>
</tr>
<tr>
<td style="text-align: left;"><code>--output</code></td>
<td style="text-align: left;"><code>output/*</code></td>
<td style="text-align: left;">输出 OM 模型路径前缀</td>
</tr>
<tr>
<td style="text-align: left;"><code>--soc_version</code></td>
<td style="text-align: left;"><code>Ascend310B4</code></td>
<td style="text-align: left;">目标芯片型号（须与香橙派一致）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--input_shape</code></td>
<td style="text-align: left;"><code>"input:1,1,28,28"</code></td>
<td style="text-align: left;">输入节点名:形状（节点名须与 ONNX 导出时一致）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--log</code></td>
<td style="text-align: left;"><code>error</code></td>
<td style="text-align: left;">只输出错误日志</td>
</tr>
</table>

**使用方法**：
```bash
cd code
bash atc_convert.sh              # 默认 SOC 为 Ascend310B4
bash atc_convert.sh Ascend310B4  # 显式指定 SOC 版本
```

### 3.3 `acl_classifier.py` — ACL 推理分类器封装

**功能**：封装 AscendCL (ACL) 接口，提供 `ACLClassifier` 类用于加载 OM 模型并执行 NPU 推理。是所有推理脚本的核心依赖模块。

**`ACLClassifier` 类主要方法**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方法</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>__init__(om_path, device_id=0)</code></td>
<td style="text-align: left;">保存模型路径和设备ID</td>
</tr>
<tr>
<td style="text-align: left;"><code>init()</code></td>
<td style="text-align: left;">初始化 ACL 运行时（acl.init + set_device + create_context）</td>
</tr>
<tr>
<td style="text-align: left;"><code>load_model()</code></td>
<td style="text-align: left;">加载 OM 模型，查询输入/输出规格，申请 Device 内存</td>
</tr>
<tr>
<td style="text-align: left;"><code>infer(input_data)</code></td>
<td style="text-align: left;">执行单次推理，返回 (输出logits, 推理耗时ms)</td>
</tr>
<tr>
<td style="text-align: left;"><code>predict(input_data)</code></td>
<td style="text-align: left;">推理并返回 (预测类别, 置信度, 耗时ms)</td>
</tr>
<tr>
<td style="text-align: left;"><code>release()</code></td>
<td style="text-align: left;">释放所有 ACL 资源（按逆序释放）</td>
</tr>
</table>

**内部实现 ACL 七阶段**：
1. `acl.init()` + `acl.rt.set_device()` — 初始化
2. `acl.mdl.load_from_mem()` — 加载 OM 模型
3. `acl.mdl.get_desc()` — 查询输入/输出规格
4. `acl.rt.malloc()` — 申请 Device 侧内存
5. `acl.rt.memcpy(H2D)` — Host→Device 数据拷贝
6. `acl.mdl.execute()` — 执行 NPU 推理
7. `acl.rt.memcpy(D2H)` — Device→Host 结果拷贝

### 3.4 `preprocess.py` — 图像预处理模块

**功能**：将手写数字图片预处理为模型推理所需的输入格式，与训练时的预处理保持一致。

**预处理流程**：
1. 读取图片并转为灰度（PIL convert('L')）
2. 缩放到 28×28（MNIST 标准尺寸）
3. 像素值归一化到 [0, 1]（除以 255.0）
4. 使用 MNIST 均值/标准差标准化：`(x - 0.1307) / 0.3081`
5. 调整为 NCHW 格式 `(1, 1, 28, 28)`

**主要函数**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">函数</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>load_image(path)</code></td>
<td style="text-align: left;">加载图片为 numpy 灰度数组</td>
</tr>
<tr>
<td style="text-align: left;"><code>preprocess_image(image)</code></td>
<td style="text-align: left;">预处理为模型输入格式 (1,1,28,28)</td>
</tr>
<tr>
<td style="text-align: left;"><code>parse_label_from_filename(name)</code></td>
<td style="text-align: left;">从文件名解析真实标签</td>
</tr>
</table>

### 3.5 `infer_single.py` — 单张图片识别脚本

**功能**：加载 OM 模型，对**单张**手写数字图片进行识别，打印识别结果（实际数字、识别数字、置信度、推理耗时）。

**使用方法**：
```bash
cd code
python3 infer_single.py ../output/simplecnn_mnist_fp32.om \
                       ../output/test_images/test_00_label0.png
```

**预期输出格式**：
```
=======================================================
单张数字图片识别结果
=======================================================
  OM 模型:     ../output/simplecnn_mnist_fp32.om
  图片文件:    test_00_label0.png
  实际数字:    0
  识别数字:    0
  识别结果:    正确
  置信度:      99.87%
  推理耗时:    0.18 ms
-------------------------------------------------------
各类别概率分布:
  数字 0:  99.87% |################################################## <-- 预测
  数字 1:   0.01% |
  ...
=======================================================
```

### 3.6 `infer_batch.py` — 批量图片识别脚本

**功能**：加载 OM 模型，对 `output/test_images/` 目录下**所有**测试图片进行批量识别，打印每张图片的识别结果，统计识别成功率，按数字类别分别统计，生成 JSON 格式的汇总报告。

**使用方法**：
```bash
cd code
python3 infer_batch.py ../output/simplecnn_mnist_fp32.om \
                       ../output/test_images/
```

**预期输出格式**：
```
=================================================================
批量数字图片识别 (共 20 张)
OM 模型: ../output/simplecnn_mnist_fp32.om
=================================================================
序号  图片文件                    实际数字  识别数字  结果    耗时(ms)
-----------------------------------------------------------------
1     test_00_label0.png          0         0         正确    0.17
2     test_01_label0.png          0         0         正确    0.16
3     test_02_label1.png          1         1         正确    0.18
...
20    test_19_label9.png          9         9         正确    0.17
-----------------------------------------------------------------
识别成功率: 20/20 = 100.0%
总耗时:     3.42 ms
平均耗时:   0.17 ms/张
吞吐率:     5847.9 张/秒
=================================================================

按数字类别统计:
数字    总数    正确    错误    成功率
------------------------------------------
0       2       2       0       100.0%
1       2       2       0       100.0%
...
9       2       2       0       100.0%
==========================================

[OK] 所有图片识别正确！

结果已保存: ../output/test_images/batch_result.json
```

### 3.7 `requirements.txt` — Python 依赖列表

**功能**：声明香橙派上需要的 Python 依赖库。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">依赖</th>
<th style="text-align: left;">版本要求</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;"><2</td>
<td style="text-align: left;">数值计算（ACL 兼容 NumPy 1.x）</td>
</tr>
<tr>
<td style="text-align: left;">Pillow</td>
<td style="text-align: left;">>=9.0.0</td>
<td style="text-align: left;">图像读取与处理</td>
</tr>
</table>

---

## 步骤四：ATC 模型转换（ONNX → OM）

> 在香橙派开发板上执行 ATC 命令，将 `models/` 目录下的 ONNX 模型转换为 `output/` 目录下的 OM 模型。

### 4.1 ATC 转换原理

ATC（Ascend Tensor Compiler）是 CANN 工具链中的模型编译工具，其工作流程：

```text
ONNX模型 → 解析算子图 → 算子映射到NPU → 图融合优化 → 内存规划 → 生成OM指令流
```

编译过程中 ATC 完成以下优化：
- **算子映射**：将 ONNX 标准算子映射到昇腾硬件的最优实现
- **图融合**：将相邻算子（如 Conv+ReLU）融合为单算子，减少中间结果内存搬运
- **内存规划**：分析张量生命周期，复用不重叠的中间结果内存

### 4.2 执行 ATC 转换

**方式一：使用转换脚本（推荐）**

```bash
cd code
source setup_env.sh          # 加载 CANN 环境
bash atc_convert.sh          # 执行 ATC 转换
```

**方式二：手动执行 ATC 命令**

```bash
# 转换 FP32 全精度模型
atc --framework=5 \
    --model=models/simplecnn_mnist_fp32.onnx \
    --output=output/simplecnn_mnist_fp32 \
    --soc_version=Ascend310B4 \
    --input_shape="input:1,1,28,28" \
    --log=error

# 转换剪枝优化模型
atc --framework=5 \
    --model=models/simplecnn_mnist_pruned.onnx \
    --output=output/simplecnn_mnist_pruned \
    --soc_version=Ascend310B4 \
    --input_shape="input:1,1,28,28" \
    --log=error
```

**ATC 命令参数说明**：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">值</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>--framework=5</code></td>
<td style="text-align: left;">5</td>
<td style="text-align: left;">输入框架类型：5=ONNX</td>
</tr>
<tr>
<td style="text-align: left;"><code>--model</code></td>
<td style="text-align: left;">ONNX 文件路径</td>
<td style="text-align: left;">输入 ONNX 模型</td>
</tr>
<tr>
<td style="text-align: left;"><code>--output</code></td>
<td style="text-align: left;">输出路径前缀</td>
<td style="text-align: left;">输出 OM 文件路径（自动加 .om 后缀）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--soc_version</code></td>
<td style="text-align: left;"><code>Ascend310B4</code></td>
<td style="text-align: left;">目标芯片型号，须与香橙派实际芯片一致</td>
</tr>
<tr>
<td style="text-align: left;"><code>--input_shape</code></td>
<td style="text-align: left;"><code>"input:1,1,28,28"</code></td>
<td style="text-align: left;">输入节点名:形状，节点名须与 ONNX 导出时一致</td>
</tr>
<tr>
<td style="text-align: left;"><code>--log=error</code></td>
<td style="text-align: left;">error</td>
<td style="text-align: left;">日志级别，只输出错误信息</td>
</tr>
</table>

> **关键注意**：`--input_shape` 中的节点名 `input` 必须与实验5.3导出 ONNX 时 `torch.onnx.export()` 中 `input_names=['input']` 指定的名称完全一致，否则 ATC 会报 `input_shape not match` 错误。

### 4.3 预期结果

ATC 转换成功后输出：`ATC run success, welcome to the next use.`

```text
============================================================
ATC 模型转换 (ONNX -> OM)
目标芯片: Ascend310B4
============================================================

[1] 转换: simplecnn_mnist_fp32.onnx -> simplecnn_mnist_fp32.om
    描述: FP32 全精度模型
ATC run success, welcome to the next use.
    [OK] OM 模型已生成: ../output/simplecnn_mnist_fp32.om (916K)

[2] 转换: simplecnn_mnist_pruned.onnx -> simplecnn_mnist_pruned.om
    描述: 剪枝优化模型
ATC run success, welcome to the next use.
    [OK] OM 模型已生成: ../output/simplecnn_mnist_pruned.om (532K)

============================================================
转换完成: 2/2 成功
============================================================
-rw-r--r-- 1 user user 916K  output/simplecnn_mnist_fp32.om
-rw-r--r-- 1 user user 532K  output/simplecnn_mnist_pruned.om
```

**香橙派实际运行结果截图**：

![ATC模型转换实际运行结果](images/atc_convert_result.png)

**转换耗时**：每个模型约 1-3 分钟（香橙派上），期间 ATC 在做算子映射、图融合、内存规划等编译优化。

### 4.4 模型大小对比

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模型</th>
<th style="text-align: left;">ONNX 大小</th>
<th style="text-align: left;">OM 大小</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">FP32 全精度</td>
<td style="text-align: left;">~1.6 MB</td>
<td style="text-align: left;">~0.9 MB</td>
<td style="text-align: left;">OM 略小（常量折叠+权重优化）</td>
</tr>
<tr>
<td style="text-align: left;">剪枝优化</td>
<td style="text-align: left;">~0.8 MB</td>
<td style="text-align: left;">~0.5 MB</td>
<td style="text-align: left;">剪枝后参数减少，体积更小</td>
</tr>
</table>

---

## 步骤五：单张手写数字图片识别

> 使用 OM 模型对 `output/test_images/` 目录下的**单张**手写数字图片进行识别。

### 5.1 识别流程

```text
读取图片 → 灰度化 → 缩放28×28 → 归一化[0,1] → 标准化 → NCHW格式
    → ACL加载OM → Host→Device拷贝 → NPU推理 → Device→Host拷贝 → argmax → 识别结果
```

### 5.2 执行单张图片识别

```bash
cd code

# 识别数字 0 的测试图片
python3 infer_single.py ../output/simplecnn_mnist_fp32.om \
                       ../output/test_images/test_00_label0.png

# 识别数字 3 的测试图片
python3 infer_single.py ../output/simplecnn_mnist_fp32.om \
                       ../output/test_images/test_06_label3.png

# 识别数字 7 的测试图片
python3 infer_single.py ../output/simplecnn_mnist_fp32.om \
                       ../output/test_images/test_14_label7.png

# 使用剪枝模型识别
python3 infer_single.py ../output/simplecnn_mnist_pruned.om \
                       ../output/test_images/test_08_label4.png
```

### 5.3 预期输出示例

**识别 test_06_label3.png（实际数字 3）**：

```text
[OK] OM 模型已加载: ../output/simplecnn_mnist_fp32.om
     input_size=3136, output_size=40

=======================================================
单张数字图片识别结果
=======================================================
  OM 模型:     ../output/simplecnn_mnist_fp32.om
  图片文件:    test_06_label3.png
  实际数字:    3
  识别数字:    3
  识别结果:    正确
  置信度:      99.95%
  推理耗时:    0.17 ms
-------------------------------------------------------
各类别概率分布:
  数字 0:   0.00% |
  数字 1:   0.00% |
  数字 2:   0.01% |
  数字 3:  99.95% |################################################## <-- 预测
  数字 4:   0.00% |
  数字 5:   0.01% |
  数字 6:   0.00% |
  数字 7:   0.00% |
  数字 8:   0.02% |
  数字 9:   0.00% |
=======================================================
[OK] 资源已释放: ../output/simplecnn_mnist_fp32.om
```

**香橙派实际运行结果截图**：

![单张数字图片识别实际运行结果](images/infer_single_result.png)

**结果解读**：
- **实际数字 3**：从文件名 `test_06_label3.png` 中的 `label3` 解析得到
- **识别数字 3**：OM 模型推理输出的 10 维向量中，索引 3 的值最大
- **识别结果 正确**：预测值与真实标签一致
- **置信度 99.95%**：softmax 后数字 3 的概率，表明模型非常确信
- **推理耗时 0.17 ms**：NPU 推理仅 0.17 毫秒，体现了 NPU 硬件加速的极致性能

---

## 步骤六：所有测试数字图片批量识别

> 使用 OM 模型对 `output/test_images/` 目录下**所有 20 张**测试图片进行批量识别，统计识别成功率。

### 6.1 执行批量识别

```bash
cd code

# 使用 FP32 全精度模型识别所有测试图片
python3 infer_batch.py ../output/simplecnn_mnist_fp32.om \
                       ../output/test_images/

# 使用剪枝优化模型识别所有测试图片
python3 infer_batch.py ../output/simplecnn_mnist_pruned.om \
                       ../output/test_images/
```

### 6.2 预期输出示例

```text
[OK] OM 模型已加载: ../output/simplecnn_mnist_fp32.om
     input_size=3136, output_size=40

=================================================================
批量数字图片识别 (共 20 张)
OM 模型: ../output/simplecnn_mnist_fp32.om
=================================================================
序号  图片文件                    实际数字  识别数字  结果    耗时(ms)
-----------------------------------------------------------------
1     test_00_label0.png          0         0         正确    0.17
2     test_01_label0.png          0         0         正确    0.16
3     test_02_label1.png          1         1         正确    0.18
4     test_03_label1.png          1         1         正确    0.17
5     test_04_label2.png          2         2         正确    0.17
6     test_05_label2.png          2         2         正确    0.16
7     test_06_label3.png          3         3         正确    0.18
8     test_07_label3.png          3         3         正确    0.17
9     test_08_label4.png          4         4         正确    0.17
10    test_09_label4.png          4         4         正确    0.16
11    test_10_label5.png          5         5         正确    0.18
12    test_11_label5.png          5         5         正确    0.17
13    test_12_label6.png          6         6         正确    0.17
14    test_13_label6.png          6         6         正确    0.16
15    test_14_label7.png          7         7         正确    0.18
16    test_15_label7.png          7         7         正确    0.17
17    test_16_label8.png          8         8         正确    0.17
18    test_17_label8.png          8         8         正确    0.16
19    test_18_label9.png          9         9         正确    0.18
20    test_19_label9.png          9         9         正确    0.17
-----------------------------------------------------------------
识别成功率: 20/20 = 100.0%
总耗时:     3.42 ms
平均耗时:   0.17 ms/张
吞吐率:     5847.9 张/秒
=================================================================

按数字类别统计:
数字    总数    正确    错误    成功率
------------------------------------------
0       2       2       0       100.0%
1       2       2       0       100.0%
2       2       2       0       100.0%
3       2       2       0       100.0%
4       2       2       0       100.0%
5       2       2       0       100.0%
6       2       2       0       100.0%
7       2       2       0       100.0%
8       2       2       0       100.0%
9       2       2       0       100.0%
==========================================

[OK] 所有图片识别正确！

结果已保存: ../output/test_images/batch_result.json
[OK] 资源已释放: ../output/simplecnn_mnist_fp32.om
```

**香橙派实际运行结果截图**：

![批量数字图片识别实际运行结果](images/infer_batch_result.png)

### 6.3 识别结果汇总表

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">图片文件</th>
<th style="text-align: left;">实际数字</th>
<th style="text-align: left;">识别数字</th>
<th style="text-align: left;">结果</th>
<th style="text-align: left;">置信度</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">test_00_label0.png</td>
<td style="text-align: left;">0</td>
<td style="text-align: left;">0</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.8%</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">test_01_label0.png</td>
<td style="text-align: left;">0</td>
<td style="text-align: left;">0</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.9%</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">test_02_label1.png</td>
<td style="text-align: left;">1</td>
<td style="text-align: left;">1</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.7%</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">test_03_label1.png</td>
<td style="text-align: left;">1</td>
<td style="text-align: left;">1</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.8%</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">test_04_label2.png</td>
<td style="text-align: left;">2</td>
<td style="text-align: left;">2</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.9%</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">test_05_label2.png</td>
<td style="text-align: left;">2</td>
<td style="text-align: left;">2</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.6%</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">test_06_label3.png</td>
<td style="text-align: left;">3</td>
<td style="text-align: left;">3</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.9%</td>
</tr>
<tr>
<td style="text-align: left;">8</td>
<td style="text-align: left;">test_07_label3.png</td>
<td style="text-align: left;">3</td>
<td style="text-align: left;">3</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.8%</td>
</tr>
<tr>
<td style="text-align: left;">9</td>
<td style="text-align: left;">test_08_label4.png</td>
<td style="text-align: left;">4</td>
<td style="text-align: left;">4</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.7%</td>
</tr>
<tr>
<td style="text-align: left;">10</td>
<td style="text-align: left;">test_09_label4.png</td>
<td style="text-align: left;">4</td>
<td style="text-align: left;">4</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.9%</td>
</tr>
<tr>
<td style="text-align: left;">11</td>
<td style="text-align: left;">test_10_label5.png</td>
<td style="text-align: left;">5</td>
<td style="text-align: left;">5</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.8%</td>
</tr>
<tr>
<td style="text-align: left;">12</td>
<td style="text-align: left;">test_11_label5.png</td>
<td style="text-align: left;">5</td>
<td style="text-align: left;">5</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.6%</td>
</tr>
<tr>
<td style="text-align: left;">13</td>
<td style="text-align: left;">test_12_label6.png</td>
<td style="text-align: left;">6</td>
<td style="text-align: left;">6</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.9%</td>
</tr>
<tr>
<td style="text-align: left;">14</td>
<td style="text-align: left;">test_13_label6.png</td>
<td style="text-align: left;">6</td>
<td style="text-align: left;">6</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.8%</td>
</tr>
<tr>
<td style="text-align: left;">15</td>
<td style="text-align: left;">test_14_label7.png</td>
<td style="text-align: left;">7</td>
<td style="text-align: left;">7</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.7%</td>
</tr>
<tr>
<td style="text-align: left;">16</td>
<td style="text-align: left;">test_15_label7.png</td>
<td style="text-align: left;">7</td>
<td style="text-align: left;">7</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.9%</td>
</tr>
<tr>
<td style="text-align: left;">17</td>
<td style="text-align: left;">test_16_label8.png</td>
<td style="text-align: left;">8</td>
<td style="text-align: left;">8</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.8%</td>
</tr>
<tr>
<td style="text-align: left;">18</td>
<td style="text-align: left;">test_17_label8.png</td>
<td style="text-align: left;">8</td>
<td style="text-align: left;">8</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.9%</td>
</tr>
<tr>
<td style="text-align: left;">19</td>
<td style="text-align: left;">test_18_label9.png</td>
<td style="text-align: left;">9</td>
<td style="text-align: left;">9</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.6%</td>
</tr>
<tr>
<td style="text-align: left;">20</td>
<td style="text-align: left;">test_19_label9.png</td>
<td style="text-align: left;">9</td>
<td style="text-align: left;">9</td>
<td style="text-align: left;">正确</td>
<td style="text-align: left;">99.8%</td>
</tr>
</table>

**识别成功率：20/20 = 100.0%**

---

## 步骤七：编译与部署详细流程

> 以下是在香橙派开发板上从零开始部署的完整流程。

### 7.1 部署流程总览

```text
┌─────────────────────────────────────────────────────────────┐
│ 1. 环境准备    │ 加载 CANN 环境、安装 Python 依赖            │
│ 2. 文件传输    │ 将 ONNX 模型和测试图片传输到香橙派           │
│ 3. ATC 转换    │ ONNX → OM（atc_convert.sh）                 │
│ 4. 单张识别    │ 运行 infer_single.py 测试单张图片            │
│ 5. 批量识别    │ 运行 infer_batch.py 测试所有图片             │
└─────────────────────────────────────────────────────────────┘
```

### 7.2 详细部署步骤

**第1步：加载 CANN 环境**
```bash
cd ~/Lab5_4/code
source setup_env.sh
# 预期输出:
# [OK] CANN Toolkit 环境已加载: /usr/local/Ascend/ascend-toolkit
# [OK] 已限制 ATC 单进程编译 (TE_PARALLEL_COMPILER=1)
# [OK] PYTHONPATH 已配置
```

**第2步：安装 Python 依赖**
```bash
pip3 install -r requirements.txt
# 预期输出:
# Collecting numpy<2
# Collecting Pillow>=9.0.0
# Successfully installed numpy-1.24.x Pillow-9.x.x
```

**第3步：验证文件完整性**
```bash
# 检查 ONNX 模型是否存在
ls -lh ../models/*.onnx
# 预期输出:
# -rw-r--r-- 1 user user 1.6M models/simplecnn_mnist_fp32.onnx
# -rw-r--r-- 1 user user 809K models/simplecnn_mnist_pruned.onnx

# 检查测试图片是否存在
ls ../output/test_images/test_*_label*.png | wc -l
# 预期输出: 20
```

**第4步：执行 ATC 模型转换**
```bash
bash atc_convert.sh
# 预期输出:
# [1] 转换: simplecnn_mnist_fp32.onnx -> simplecnn_mnist_fp32.om
# ATC run success, welcome to the next use.
#     [OK] OM 模型已生成: ../output/simplecnn_mnist_fp32.om (916K)
# [2] 转换: simplecnn_mnist_pruned.onnx -> simplecnn_mnist_pruned.om
# ATC run success, welcome to the next use.
#     [OK] OM 模型已生成: ../output/simplecnn_mnist_pruned.om (532K)
# 转换完成: 2/2 成功
```

**第5步：单张图片识别**
```bash
python3 infer_single.py ../output/simplecnn_mnist_fp32.om \
                       ../output/test_images/test_00_label0.png
# 预期输出: 实际数字=0, 识别数字=0, 识别结果=正确
```

**第6步：批量图片识别**
```bash
python3 infer_batch.py ../output/simplecnn_mnist_fp32.om \
                       ../output/test_images/
# 预期输出: 识别成功率: 20/20 = 100.0%
```

### 7.3 一键部署命令序列

```bash
# 在香橙派上依次执行以下命令
cd ~/Lab5_4/code
source setup_env.sh                                              # 1. 加载环境
pip3 install -r requirements.txt                                 # 2. 安装依赖
bash atc_convert.sh                                              # 3. ATC转换
python3 infer_single.py ../output/simplecnn_mnist_fp32.om \
         ../output/test_images/test_00_label0.png                # 4. 单张识别
python3 infer_batch.py ../output/simplecnn_mnist_fp32.om \
         ../output/test_images/                                  # 5. 批量识别
```

---

## 实验结果与分析

### 1. 识别结果分析

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模型</th>
<th style="text-align: left;">测试图片数</th>
<th style="text-align: left;">识别正确数</th>
<th style="text-align: left;">识别成功率</th>
<th style="text-align: left;">平均耗时</th>
</tr>
<tr>
<td style="text-align: left;">FP32 全精度 OM</td>
<td style="text-align: left;">20</td>
<td style="text-align: left;">20</td>
<td style="text-align: left;"><strong>100.0%</strong></td>
<td style="text-align: left;">0.17 ms</td>
</tr>
<tr>
<td style="text-align: left;">剪枝优化 OM</td>
<td style="text-align: left;">20</td>
<td style="text-align: left;">20</td>
<td style="text-align: left;"><strong>100.0%</strong></td>
<td style="text-align: left;">0.15 ms</td>
</tr>
</table>

**识别结果解读**：
- **识别成功率 100%**：20 张测试图片（数字 0-9 各 2 张）全部识别正确，说明模型在 MNIST 任务上表现优异。
- **FP32 与剪枝模型识别率一致**：30% 结构化剪枝后精度损失极小，仍能正确识别所有测试图片。
- **每张图片推理仅 0.17 ms**：NPU 硬件加速使推理达到亚毫秒级，吞吐率约 5800 张/秒。

### 2. 性能分析

#### 香橙派上的性能数据（100 次推理基准测试）

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模型</th>
<th style="text-align: left;">平均耗时(ms)</th>
<th style="text-align: left;">最快(ms)</th>
<th style="text-align: left;">最慢(ms)</th>
<th style="text-align: left;">吞吐率(张/秒)</th>
</tr>
<tr>
<td style="text-align: left;">FP32 全精度 OM</td>
<td style="text-align: left;">0.17</td>
<td style="text-align: left;">0.16</td>
<td style="text-align: left;">0.34</td>
<td style="text-align: left;">~5,800</td>
</tr>
<tr>
<td style="text-align: left;">剪枝优化 OM</td>
<td style="text-align: left;">0.15</td>
<td style="text-align: left;">0.14</td>
<td style="text-align: left;">0.28</td>
<td style="text-align: left;">~6,600</td>
</tr>
</table>

**性能数据解读**：
- **亚毫秒级推理**：OM 模型在 NPU 上推理仅 0.17 ms，这是因为 ATC 编译时已将算子映射到 AICore 硬件、完成图融合和内存优化，推理时直接执行预编译的指令流。
- **推理耗时极稳定**：最快 0.16 ms、最慢 0.34 ms，波动极小。因为 OM 推理无需运行时图解释和算子分发，消除了框架调度的不确定性。
- **剪枝模型更快**：剪枝后参数量减少约 50%，模型结构更小，推理速度进一步提升至 0.15 ms。

#### 加速比来源分析

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层面</th>
<th style="text-align: left;">贡献</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">硬件层面</td>
<td style="text-align: left;">最大</td>
<td style="text-align: left;">CPU → NPU AICore，矩阵乘加由专用 Cube 单元完成</td>
</tr>
<tr>
<td style="text-align: left;">编译层面</td>
<td style="text-align: left;">中</td>
<td style="text-align: left;">ATC 完成图融合、算子调度、内存复用，消除框架开销</td>
</tr>
<tr>
<td style="text-align: left;">运行时层面</td>
<td style="text-align: left;">小</td>
<td style="text-align: left;">OM 预编译后直接加载执行，无需运行时图解释</td>
</tr>
</table>

### 3. 模型大小对比

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模型</th>
<th style="text-align: left;">ONNX (KB)</th>
<th style="text-align: left;">OM (KB)</th>
<th style="text-align: left;">缩减比</th>
</tr>
<tr>
<td style="text-align: left;">FP32 全精度</td>
<td style="text-align: left;">1,648</td>
<td style="text-align: left;">916</td>
<td style="text-align: left;">44.4%</td>
</tr>
<tr>
<td style="text-align: left;">剪枝优化</td>
<td style="text-align: left;">809</td>
<td style="text-align: left;">532</td>
<td style="text-align: left;">34.2%</td>
</tr>
</table>

**模型大小解读**：OM 文件比 ONNX 更小，因为 ATC 编译时做了常量折叠（将可计算的常量提前求值）和权重压缩优化。剪枝模型由于参数量减少 50%，文件体积进一步缩小。

### 4. ATC 图融合分析

ATC 编译过程中会输出图融合统计信息（`fusion_result.json`）：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">融合类别</th>
<th style="text-align: left;">规则数</th>
<th style="text-align: left;">命中次数</th>
<th style="text-align: left;">生效次数</th>
</tr>
<tr>
<td style="text-align: left;">graph_fusion</td>
<td style="text-align: left;">31</td>
<td style="text-align: left;">85</td>
<td style="text-align: left;">11</td>
</tr>
<tr>
<td style="text-align: left;">ub_fusion</td>
<td style="text-align: left;">2</td>
<td style="text-align: left;">4</td>
<td style="text-align: left;">4</td>
</tr>
</table>

**图融合解读**：
- **graph_fusion**：图级融合将相邻算子合并（如 Conv+ReLU 融合），减少算子数量和中间结果内存搬运。
- **ub_fusion**：UB（Unified Buffer）级融合将多个算子融合到同一片上高速缓冲中执行，避免中间结果写回 HBM，大幅减少内存访问延迟。

---

## 实验总结

本实验在香橙派开发板（昇腾310B NPU）上完成了模型部署的最后两个环节，完整链路如下：

```text
实验5.3 (云沙箱 910B3)              本实验 (香橙派 310B4)
PyTorch训练 → ONNX导出  ────────>  ATC转换 → OM推理 → 数字识别
```

### 关键收获

1. **ATC 是端侧部署的关键环节**：ATC 将通用的 ONNX 算子图编译为针对 Ascend310B4 硬件深度优化的 OM 指令流，完成算子映射、图融合、内存规划等编译优化。这是 OM 推理极快的根本原因。

2. **OM 推理极快且稳定**：在香橙派 NPU 上推理仅 0.17 ms/张，吞吐率约 5800 张/秒，且耗时极稳定。预编译消除了运行时开销，NPU 专用计算单元提供了极致性能。

3. **识别成功率 100%**：20 张测试图片（数字 0-9 各 2 张）全部识别正确，说明从训练到部署的全链路（PyTorch→ONNX→OM→ACL推理）是精确等价的，转换过程无损。

4. **剪枝模型在端侧优势明显**：30% 结构化剪枝后模型体积减小 34%，推理速度提升至 0.15 ms，且识别率保持 100%，非常适合资源受限的边缘设备。

5. **云训练→端部署的工程范式**：在云沙箱（910B3）上完成训练和 ONNX 导出，在边缘设备（310B4）上完成 ATC 转换和推理，体现了"云端训练、边缘推理"的典型部署范式。

### 产物清单

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">产物</th>
<th style="text-align: left;">位置</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>simplecnn_mnist_fp32.om</code></td>
<td style="text-align: left;"><code>output/</code></td>
<td style="text-align: left;">FP32 全精度 OM 离线模型</td>
</tr>
<tr>
<td style="text-align: left;"><code>simplecnn_mnist_pruned.om</code></td>
<td style="text-align: left;"><code>output/</code></td>
<td style="text-align: left;">剪枝优化 OM 模型</td>
</tr>
<tr>
<td style="text-align: left;"><code>batch_result.json</code></td>
<td style="text-align: left;"><code>output/test_images/</code></td>
<td style="text-align: left;">批量识别结果（含每张图片的预测、置信度、耗时）</td>
</tr>
<tr>
<td style="text-align: left;"><code>code/</code></td>
<td style="text-align: left;"><code>code/</code></td>
<td style="text-align: left;">全部推理程序代码（Python + Shell）</td>
</tr>
</table>

### code 目录代码清单

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">语言</th>
<th style="text-align: left;">功能</th>
</tr>
<tr>
<td style="text-align: left;"><code>setup_env.sh</code></td>
<td style="text-align: left;">Shell</td>
<td style="text-align: left;">CANN 环境初始化</td>
</tr>
<tr>
<td style="text-align: left;"><code>atc_convert.sh</code></td>
<td style="text-align: left;">Shell</td>
<td style="text-align: left;">ATC 模型转换（ONNX→OM）</td>
</tr>
<tr>
<td style="text-align: left;"><code>acl_classifier.py</code></td>
<td style="text-align: left;">Python</td>
<td style="text-align: left;">ACL 推理分类器封装（核心模块）</td>
</tr>
<tr>
<td style="text-align: left;"><code>preprocess.py</code></td>
<td style="text-align: left;">Python</td>
<td style="text-align: left;">图像预处理（归一化+标准化+NCHW）</td>
</tr>
<tr>
<td style="text-align: left;"><code>infer_single.py</code></td>
<td style="text-align: left;">Python</td>
<td style="text-align: left;">单张图片识别</td>
</tr>
<tr>
<td style="text-align: left;"><code>infer_batch.py</code></td>
<td style="text-align: left;">Python</td>
<td style="text-align: left;">批量图片识别与成功率统计</td>
</tr>
<tr>
<td style="text-align: left;"><code>requirements.txt</code></td>
<td style="text-align: left;">Text</td>
<td style="text-align: left;">Python 依赖列表</td>
</tr>
</table>

---

## 常见问题与故障排查

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">现象</th>
<th style="text-align: left;">可能原因</th>
<th style="text-align: left;">解决方法</th>
</tr>
<tr>
<td style="text-align: left;"><code>atc: command not found</code></td>
<td style="text-align: left;">CANN 环境变量未加载</td>
<td style="text-align: left;"><code>source /usr/local/Ascend/ascend-toolkit/set_env.sh</code></td>
</tr>
<tr>
<td style="text-align: left;">ATC 报 <code>--framework is required</code></td>
<td style="text-align: left;">直接输入 atc 不带参数</td>
<td style="text-align: left;">正常现象，说明工具可用</td>
</tr>
<tr>
<td style="text-align: left;">ATC 报 <code>input_shape not match</code></td>
<td style="text-align: left;">节点名不一致或形状含空格</td>
<td style="text-align: left;">核对 ONNX 导出时的 <code>input_names</code>，须为 <code>input</code></td>
</tr>
<tr>
<td style="text-align: left;">ATC 转换卡住或内存耗尽</td>
<td style="text-align: left;">并行编译进程过多</td>
<td style="text-align: left;"><code>export TE_PARALLEL_COMPILER=1</code> 和 <code>MAX_COMPILE_CORE_NUMBER=1</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>soc_version</code> 不匹配</td>
<td style="text-align: left;">芯片型号填写错误</td>
<td style="text-align: left;"><code>npu-smi info</code> 查询实际型号，香橙派为 <code>Ascend310B4</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>import acl</code> 失败</td>
<td style="text-align: left;">CANN Python 接口路径未配置</td>
<td style="text-align: left;">检查 PYTHONPATH，确认 <code>set_env.sh</code> 已 source</td>
</tr>
<tr>
<td style="text-align: left;">OM 推理结果异常</td>
<td style="text-align: left;">输入未按 NCHW 组织或未归一化</td>
<td style="text-align: left;">确认输入形状 (1,1,28,28)、dtype 为 float32</td>
</tr>
<tr>
<td style="text-align: left;">NumPy 版本冲突</td>
<td style="text-align: left;">ACL 编译于 NumPy 1.x</td>
<td style="text-align: left;"><code>pip3 install 'numpy<2'</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>No module named PIL</code></td>
<td style="text-align: left;">Pillow 未安装</td>
<td style="text-align: left;"><code>pip3 install Pillow</code></td>
</tr>
</table>

### ATC 命令参数速查

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">含义</th>
</tr>
<tr>
<td style="text-align: left;"><code>--model</code></td>
<td style="text-align: left;">输入 ONNX 模型文件路径</td>
</tr>
<tr>
<td style="text-align: left;"><code>--framework=5</code></td>
<td style="text-align: left;">输入框架类型：5=ONNX</td>
</tr>
<tr>
<td style="text-align: left;"><code>--output</code></td>
<td style="text-align: left;">输出 OM 文件路径前缀（自动加 .om）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--soc_version</code></td>
<td style="text-align: left;">目标芯片型号，须与板卡一致</td>
</tr>
<tr>
<td style="text-align: left;"><code>--input_shape</code></td>
<td style="text-align: left;">输入节点名:形状（如 <code>input:1,1,28,28</code>）</td>
</tr>
<tr>
<td style="text-align: left;"><code>--log=error</code></td>
<td style="text-align: left;">日志级别（error/warning/info/debug）</td>
</tr>
</table>

> **最易出错的参数**：`--input_shape` 的节点名必须与 ONNX 导出时 `input_names` 完全一致。`--soc_version` 必须通过 `npu-smi info` 查询实际芯片型号后填写。

---

## 关键问题探究

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">问题</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">为什么 ONNX 不能直接在 NPU 上推理，必须经过 ATC 转换为 OM？ATC 转换做了哪些优化？</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>--soc_version=Ascend310B4</code> 写成 <code>Ascend910B3</code> 会发生什么？生成的 OM 能否在香橙派上运行？</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">OM 推理为什么能达到 0.17 ms 的亚毫秒级？从硬件、编译、运行时三个层面分析</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">剪枝模型推理速度为什么比 FP32 更快？剪枝减少了哪些计算量？</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">如果输入图片未做标准化（不减均值、不除标准差），识别结果会怎样？为什么？</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">云沙箱（910B3）上生成的 OM 能否直接拿到香橙派（310B4）上运行？为什么？</td>
</tr>
</table>

**关键问题解析**：
- **问题1**：ONNX 是通用的跨框架中间格式，包含标准算子图，但未针对特定硬件优化。ATC 将 ONNX 编译为针对昇腾 NPU 深度优化的 OM 指令流，完成算子映射到 AICore、图融合（Conv+ReLU等）、内存复用规划等优化，使推理能直接在 NPU 硬件上高效执行。
- **问题2**：`--soc_version` 决定了 ATC 编译时针对的硬件型号。若写成 `Ascend910B3`，生成的 OM 包含 910B3 特有指令，在 310B4 上运行会报指令不兼容错误。**OM 模型必须与目标硬件一致**，不能跨芯片型号使用。
- **问题3**：① 硬件层面：NPU 的 AICore 有专用 Cube 矩阵计算单元，矩阵乘加在一个时钟周期完成；② 编译层面：ATC 完成图融合消除中间结果内存搬运、选择最优算子实现、规划内存复用；③ 运行时层面：OM 预编译后直接加载执行，无需运行时图解释和算子分发。
- **问题4**：结构化剪枝物理移除了 30% 的卷积通道和全连接神经元，模型参数量减少约 50%。更小的模型意味着更少的矩阵乘加运算和更少的内存访问，因此推理更快。
- **问题5**：训练时使用了标准化预处理（减均值 0.1307、除标准差 0.3081），如果推理时不做同样的标准化，输入数据分布与训练时不一致，模型会输出错误的预测结果。**推理预处理必须与训练预处理完全一致**。
- **问题6**：不能。不同芯片型号（910B3 vs 310B4）的指令集、内存层次、算子实现不同，ATC 编译时针对特定 SOC 优化。**OM 模型必须在目标设备上用正确的 `--soc_version` 重新编译**。

---

## 参考资料

- [昇腾 CANN 文档](https://www.hiascend.com/document)
- [ATC 工具使用指南](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [AscendCL API 参考](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [CANN 社区样例](https://gitee.com/ascend/samples)
- [香橙派 AIPro 开发板文档](http://www.orangepi.cn/html/hardWare/computingAndMicrocontroller/serviceAndSupport.html)